In [13]:
import pandas as pd
import numpy as np
import joblib
import os
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

np.random.seed(42)

In [ ]:
#  LOAD DATASET
FILE_NAME = '/kaggle/input/datasets/nifraw/pakwheels-dataset/real_workd_pakwheels.csv' 

if not os.path.exists(FILE_NAME):
    # This block helps you debug the path if it fails
    print("File not found! Searching directory...")
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(f"Found file: {os.path.join(dirname, filename)}")
    raise FileNotFoundError(f"Could not find {FILE_NAME}")

df = pd.read_csv(FILE_NAME)
print(f'Successfully loaded. Shape: {df.shape}')

Successfully loaded. Shape: (62302, 14)


In [ ]:
# PREPROCESSING

df.columns = df.columns.str.strip().str.lower()

df = df.rename(columns={
    'engine': 'engine_capacity_cc',
    'body':   'body_type',
    'price':  'price_pkr'
})

df['price_pkr'] = pd.to_numeric(df['price_pkr'], errors='coerce')
df = df.dropna(subset=['price_pkr'])

PRICE_THRESHOLD = df['price_pkr'].median()
df['price_category'] = (df['price_pkr'] >= PRICE_THRESHOLD).astype(int)

df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['car_age'] = 2026 - df['year']

NUMERIC_FEATURES = ['year', 'car_age', 'engine_capacity_cc', 'mileage']
CATEGORICAL_FEATURES = ['make', 'transmission', 'fuel', 'body_type', 'city']
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET   = 'price_category'

df_model = df[FEATURES + [TARGET]].dropna()
print(f'Price threshold (median): PKR {PRICE_THRESHOLD:,.0f}')
print(f'Training data shape: {df_model.shape}')

Price threshold (median): PKR 2,700,000
Training data shape: (51708, 10)


In [ ]:
#  ML PIPELINE

X = df_model[FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer,     NUMERIC_FEATURES),
    ('cat', categorical_transformer, CATEGORICAL_FEATURES),
])

svm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   SVC(kernel='rbf', C=1.0, probability=True, random_state=42)),
])

In [ ]:
#TRAIN AND EVALUATE

print('Training SVM Model...')
svm_pipeline.fit(X_train, y_train)
print('Training complete.')

y_pred = svm_pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f'\nAccuracy: {acc*100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

Training SVM Model...
Training complete.

Accuracy: 95.33%

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.95      0.95      4975
           1       0.96      0.95      0.95      5367

    accuracy                           0.95     10342
   macro avg       0.95      0.95      0.95     10342
weighted avg       0.95      0.95      0.95     10342



In [ ]:
# SAVE MODEL 

MODEL_FILE = 'pakwheels_svm_model.pkl'
METADATA_FILE = 'model_metadata.json'

# Save the entire pipeline (includes scaling and encoding)
joblib.dump(svm_pipeline, MODEL_FILE)

# Save metadata for your API/Frontend
metadata = {
    'price_threshold_pkr': float(PRICE_THRESHOLD),
    'numeric_features': NUMERIC_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'accuracy': round(acc, 4),
    'classes': ['Low Price', 'High Price']
}

with open(METADATA_FILE, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'\nFiles saved successfully:')
print(f'- {MODEL_FILE}')
print(f'- {METADATA_FILE}')





Files saved successfully:
- pakwheels_svm_model.pkl
- model_metadata.json
